In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn as nn
import copy
import torch

def training_helper(model_class,
                   train_data, val_data,
                   lr, batch_size,
                   epochs=100, patience=5,
                   arch_params=None):
  if arch_params:
    model = model_class(**arch_params)
  else:
    model = model_class()

  model.apply(initialize_weights)
  model = model.to(device)

  train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

  loss_fn = nn.CrossEntropyLoss()
  optimizer=optim.SGD(model.parameters(), lr=lr)

  #training loop with early stopping
  train_loss_hist, val_loss_hist = [], []
  train_acc_hist, val_acc_hist = [], []
  best_val_loss = float('inf')
  patience_counter = 0
  improvement_threshold = 1e-3
  best_model_weights = None

  for epoch in range(epochs):
    #Training
        model.train()
        running_loss, correct_train, total_train = 0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        avg_train_loss = running_loss / len(train_loader)
        avg_train_acc = 100 * correct_train / total_train
        train_loss_hist.append(avg_train_loss)
        train_acc_hist.append(avg_train_acc)

        #Validation
        model.eval()
        running_val_loss, correct_val, total_val = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)

                running_val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        avg_val_loss = running_val_loss / len(val_loader)
        avg_val_acc = 100 * correct_val / total_val
        val_loss_hist.append(avg_val_loss)
        val_acc_hist.append(avg_val_acc)

        # Early Stopping
        improvement = best_val_loss - avg_val_loss
        if improvement > improvement_threshold:
            best_val_loss = avg_val_loss
            patience_counter = 0
            # Save the best model state
            best_model_weights = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= patience:
            # print(f"Early stopping at epoch {epoch + 1}") # Optional print
            break

    # Load the best weights before returning
  if best_model_weights:
        model.load_state_dict(best_model_weights)

    # Return the training history and the best model (for C2 comparison)
  return train_loss_hist, val_loss_hist, train_acc_hist, val_acc_hist, model

NameError: name 'nn' is not defined

In [ ]:
#Learning Rate Analysis
import matplotlib.pyplot as plt
import pandas as pd

lrs = [0.001, 0.01, 0.1, 1.0]
lr_results = {}
baseline_batch_size = 64
baseline_epochs = 50

print("Starting Learing rate analysis")
for lr in lrs:
  train_loss, val_loss, train_acc, val_acc, _ = training_helper(
        FeedForwardNN,
        train_dataset, val_dataset,
        lr=lr,
        batch_size=baseline_batch_size,
        epochs=baseline_epochs
    )
  lr_results[lr] = {
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'epochs_ran': len(val_loss)
    }
  print("Learnin rate analysis done!")


In [ ]:
#Get the best learning rate
best_val_acc = 0.0
best_lr = 0.0

for lr, results in lr_results.items():
    current_best_acc = max(results['val_acc'])

    if current_best_acc > best_val_acc:
        best_val_acc = current_best_acc
        best_lr = lr